In [ ]:
import os, sys
sys.path.append('../')
import MeshFEM, mesh, benchmark
import numpy as np
import pickle
import matplotlib.pyplot as plt

In [ ]:
import bisect, math
import plot_video_utils
import video_writer

In [ ]:
base_path = 'local_exps_with_uv/'

In [ ]:
model_name_list = [entry.name for entry in os.scandir(base_path) if entry.is_dir()]
# delete Figs in model_name_list 
if 'Figs' in model_name_list:  model_name_list.remove('Figs')
if 'Videos' in model_name_list:  model_name_list.remove('Videos')

In [ ]:
thread_num_list = [0]
hessian_option_list = ['Adaptive', 'Always', 'Never']

In [ ]:
user_model_name = 'bear_cut'
if user_model_name not in model_name_list:
    raise RuntimeError(f"[Error] '{user_model_name}' is not in directory {base_path} !")

In [ ]:
obj_list, grad_norm_list, hessian_projected_list, hessian_shifted_amount_list = plot_video_utils.readHessianData(os.path.join(base_path, user_model_name))

In [ ]:
obj_grad_time_list = plot_video_utils.readConvergenceTimingData(os.path.join(base_path, user_model_name), thread_num_list)

In [ ]:
aligned_timing_list = plot_video_utils.alignTiming(obj_grad_time_list, grad_norm_list)

In [ ]:
max_num_steps = max(grad_norm_list[0].shape[0], grad_norm_list[1].shape[0], grad_norm_list[2].shape[0])
max_grad_norm = max(np.max(grad_norm_list[0]), np.max(grad_norm_list[1]), np.max(grad_norm_list[2]))
min_grad_norm = min(np.min(grad_norm_list[0]), np.min(grad_norm_list[1]), np.min(grad_norm_list[2]))

max_N_grad_norm = math.ceil(math.log10(max_grad_norm))
min_N_grad_norm = math.floor(math.log10(min_grad_norm))

max_obj = max(np.max(obj_list[0]), np.max(obj_list[1]), np.max(obj_list[2]))
min_obj = min(np.min(obj_list[0]), np.min(obj_list[1]), np.min(obj_list[2]))

max_N_obj = math.ceil(math.log10(max_obj))
min_N_obj = math.floor(math.log10(min_obj))


In [ ]:
video_folder = 'Videos'
video_dir = os.path.join(base_path, video_folder)
if not os.path.exists(video_dir):  os.makedirs(video_dir)

In [ ]:
hessian_projected_list

In [ ]:
brek

# Gradient norm vs Iter video

In [ ]:
grad_iter_vdname = user_model_name + '_GradVSIter.mp4'
fig = plt.figure(figsize=(8, 8))

plt.xlim(0, max_num_steps)
plt.ylim(10**(min_N_grad_norm), 10**(max_N_grad_norm))

plt.title(f"Model: {user_model_name}", fontsize=16)
plt.yscale('log')
plt.xlabel("Iteration", fontsize=12)
plt.ylabel(" Grad Norm ", fontsize=14)
plt.legend()

pw = video_writer.PlotVideoWriter(os.path.join(video_dir, grad_iter_vdname), plt.gcf(), dpi=300, )

In [ ]:
iterations_list = []
for i in range(3):
    iterations = np.arange(0, grad_norm_list[i].shape[0])
    iterations_list.append(iterations)
color_list = ['dodgerblue', 'magenta', 'tomato']
line_style_list = ['-', '--', '-.']

fps = 30
spf = 1 / fps
totalTime = max(aligned_timing_list[0][-1], aligned_timing_list[1][-1], aligned_timing_list[2][-1])
numFrames = int(math.ceil(totalTime / spf))

adaptive_projtrue_iter = iterations_list[0][hessian_projected_list[0]==1]
grad_norm_projtrue_list = grad_norm_list[0][hessian_projected_list[0]==1]

for f in range(numFrames):
    frameTime = f * spf
    fig = plt.figure(figsize=(8, 8))
    index_for_dots = 0
    for i in range(3):
        iterationForFrame = max(0, bisect.bisect_right(aligned_timing_list[i], frameTime) - 1)
        plt.plot(iterations_list[i][:iterationForFrame+1], grad_norm_list[i][:iterationForFrame+1], 
                 ls=line_style_list[i], color=color_list[i], label=hessian_option_list[i])
        if i == 0: index_for_dots = max(0, bisect.bisect_left(adaptive_projtrue_iter, iterationForFrame))
        plt.scatter(adaptive_projtrue_iter[:index_for_dots], grad_norm_projtrue_list[:index_for_dots], color='blue', marker='o', s=60)
    
    plt.xlim(0, max_num_steps)
    plt.ylim(10**(min_N_grad_norm), 10**(max_N_grad_norm))

    plt.title(f"Model: {user_model_name}", fontsize=16)
    plt.yscale('log')
    plt.xlabel("Iteration", fontsize=12)
    plt.ylabel(" Grad Norm ", fontsize=14)
    plt.legend(loc="upper right")
    print(pw.frameDataSize())
    print(len(plt.gcf().ravel()))
    
    brek
    pw.writeFrame(plt.gcf())
    plt.close()

pw.finish()
    
    
    

In [ ]:
brek

# Energy vs Iter Video

In [ ]:
obj_iter_vdname = user_model_name + '_ObjVSIter.mp4'
fig = plt.figure(figsize=(8, 8))

plt.xlim(0, max_num_steps)
plt.ylim(10**(min_N_obj), 10**(max_N_obj))

plt.title(f"Model: {user_model_name}", fontsize=16)
plt.yscale('log')
plt.xlabel("Iteration", fontsize=12)
plt.ylabel(" Energy ", fontsize=14)
plt.legend()

pw = video_writer.PlotVideoWriter(os.path.join(video_dir, obj_iter_vdname), plt.gcf(), dpi=300, )

In [ ]:
iterations_list = []
for i in range(3):
    iterations = np.arange(0, obj_list[i].shape[0])
    iterations_list.append(iterations)
color_list = ['dodgerblue', 'magenta', 'tomato']
line_style_list = ['-', '--', '-.']

fps = 30
spf = 1 / fps
totalTime = max(aligned_timing_list[0][-1], aligned_timing_list[1][-1], aligned_timing_list[2][-1])
numFrames = int(math.ceil(totalTime / spf))

adaptive_projtrue_iter = iterations_list[0][hessian_projected_list[0]==1]
obj_projtrue_list = obj_list[0][hessian_projected_list[0]==1]

for f in range(numFrames):
    frameTime = f * spf
    fig = plt.figure(figsize=(8, 8))
    for i in range(3):
        iterationForFrame = max(0, bisect.bisect_right(aligned_timing_list[i], frameTime) - 1)
        plt.plot(iterations_list[i][:iterationForFrame+1], obj_list[i][:iterationForFrame+1], 
                 ls=line_style_list[i], color=color_list[i], label=hessian_option_list[i])
        if i == 0: index_for_dots = max(0, bisect.bisect_left(adaptive_projtrue_iter, iterationForFrame))
        plt.scatter(adaptive_projtrue_iter[:index_for_dots], obj_projtrue_list[:index_for_dots], color='blue', marker='o', s=60)
    
    plt.xlim(0, max_num_steps)
    plt.ylim(10**(min_N_obj), 10**(max_N_obj))

    plt.title(f"Model: {user_model_name}", fontsize=16)
    plt.yscale('log')
    plt.xlabel("Iteration", fontsize=12)
    plt.ylabel(" Energy ", fontsize=14)
    plt.legend(loc="upper right")
    pw.writeFrame(plt.gcf())
    plt.close()

pw.finish()